# End-of-Day Mini Project — Student Version (Google Colab)
## Predict California Housing Prices with Neural Networks (PyTorch)

This notebook is a **student-learning version** of the attached mini project. It is organized as a **Google Colab-ready Jupyter notebook** and preserves the project flow and topics from the lab guide while adding explanations of **what you are learning** and **why each step matters**.

## Lab overview
In this mini project, you will build a complete deep learning regression workflow in PyTorch to predict California housing prices using the California Housing dataset. You will move through the full pipeline: data loading, preprocessing, model creation, training, visualization, model comparison, and saving the trained model.

### In this project, you will
- Load and preprocess the California Housing dataset
- Build a neural network model in PyTorch
- Implement a training loop with loss logging
- Visualize training and validation losses
- Compare the neural network with a baseline Linear Regression model
- Analyze model predictions visually
- Save and (optionally) reload trained models in PyTorch

### Estimated completion time
- **50 minutes**

### Runtime type
- Set runtime type to **CPU**


# Task 0 (Colab setup): Imports and environment check

### What you are learning
This setup cell prepares the notebook environment and confirms the PyTorch version/runtime. It is good practice to verify your environment before running a project.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Runtime recommendation: CPU")


# Task 1: Loading and preparing data

### What you are learning
You are learning how to prepare a real-world tabular dataset for deep learning: load the data, split train/test, normalize features, and convert arrays to PyTorch tensors. These steps improve training stability and make the data compatible with PyTorch models.

### Steps (from the lab)
1. Import the necessary libraries.
2. Load the dataset and inspect shape/feature names/sample targets.
3. Split the data into train and test sets.
4. Normalize features using `StandardScaler`.
5. Convert the arrays to PyTorch tensors and reshape targets with `.view(-1, 1)`.


In [ ]:
# 1. Import the necessary libraries
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

# 2. Load the dataset and inspect it
housing = fetch_california_housing()
X, y = housing.data, housing.target

print("Dataset Shape:", X.shape)
print("Feature Names:", housing.feature_names)
print("Sample target values (first 5):", y[:5])


In [ ]:
# 3. Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")


In [ ]:
# 4. Normalize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature mean after normalization (first 3, should be ~0):", X_train_scaled.mean(axis=0)[:3])
print("Feature std after normalization (first 3, should be ~1):", X_train_scaled.std(axis=0)[:3])


In [ ]:
# 5. Convert to PyTorch tensors (targets reshaped for regression output shape compatibility)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("PyTorch Tensor Shapes:")
print("X_train_tensor:", X_train_tensor.shape)
print("X_test_tensor :", X_test_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)
print("y_test_tensor :", y_test_tensor.shape)


### Task 1 explanation (learning takeaways)
- Train/test splitting supports honest evaluation on unseen data.
- Feature normalization improves convergence and training stability.
- Tensor conversion is required for PyTorch training.
- Reshaping targets to `(N, 1)` matches the neural network regression output shape.


# Task 2: Defining a neural network model

### What you are learning
You are learning how to build a custom PyTorch model using `nn.Module`. The lab uses a 2-hidden-layer MLP with ReLU activations and a single output node for regression.

### Steps (from the lab)
1. Import `torch.nn as nn`.
2. Define a custom neural network class (`HousingMLP`).
3. Define the architecture using `nn.Sequential` (8→64→32→1 with ReLU activations).
4. Define the forward pass.
5. Create an instance of the model.
6. Print the model architecture to verify it.


In [ ]:
# 1. Import torch.nn
import torch.nn as nn

# 2. Define a custom neural network (MLP) class
class HousingMLP(nn.Module):
    def __init__(self):
        # 2. Initialize base class
        super(HousingMLP, self).__init__()

        # 3. Define architecture
        self.model = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    # 4. Define forward pass
    def forward(self, x):
        return self.model(x)

# 5. Create model instance
model = HousingMLP()

# 6. Print model architecture
print("Neural Network Model Defined:\n", model)


### Task 2 explanation (learning takeaways)
- `nn.Module` is PyTorch’s base class for trainable models.
- `nn.Sequential` makes the layer stack easy to read.
- ReLU gives the network nonlinear modeling capability.
- A single output node is used because the target is a continuous value (regression).


# Task 3: Implementing a training loop

### What you are learning
This task covers the core deep learning training workflow: define loss + optimizer, run forward/backward passes, update weights, track training and validation loss, and visualize learning curves.

### Steps (from the lab)
1. Import `torch.optim`.
2. Define the loss function and optimizer.
3. Set training epochs and initialize tracking lists.
4. Start the training loop and set model to training mode.
   - 4.1 Clear gradients
   - 4.2 Forward pass on training data
   - 4.3 Compute training loss
   - 4.4 Backward pass
   - 4.5 Update parameters
5. Set model to evaluation mode for validation (`torch.no_grad()`).
   - 5.1 Forward pass on test data
   - 5.2 Compute validation loss
6. Store losses for analysis.
7. Print progress every 10 epochs.
8. Visualize the loss curve.
   - 8.1 Plot training and validation losses
   - 8.2 Review the loss plot


In [ ]:
# 1. Import optimizer module
import torch.optim as optim

# 2. Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss Function: MSELoss (for regression)")
print("Optimizer: Adam")
print("Learning Rate:", optimizer.defaults['lr'])


In [ ]:
# 3. Set epochs and loss trackers
num_epochs = 100
train_losses = []
val_losses = []

# 4. Training loop
for epoch in range(num_epochs):
    model.train()  # 4. set training mode

    # 4.1 clear gradients
    optimizer.zero_grad()

    # 4.2 forward pass on training data
    outputs = model(X_train_tensor)

    # 4.3 compute training loss
    loss = criterion(outputs, y_train_tensor)

    # 4.4 backward pass
    loss.backward()

    # 4.5 update parameters
    optimizer.step()

    # 5. validation phase
    model.eval()
    with torch.no_grad():
        # 5.1 forward pass on test data
        val_outputs = model(X_test_tensor)

        # 5.2 compute validation loss
        val_loss = criterion(val_outputs, y_test_tensor)

    # 6. store losses
    train_losses.append(loss.item())
    val_losses.append(val_loss.item())

    # 7. display progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")


In [ ]:
# 8. Visualize the loss curve
# 8.1 Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

# 8.2 Review:
# Decreasing curves generally indicate learning.
# A large train/val gap may suggest overfitting.


### Task 3 explanation (learning takeaways)
- `MSELoss` measures squared error and is standard for regression.
- `Adam` updates model weights using adaptive learning rates.
- `model.train()` / `model.eval()` are important mode switches in PyTorch.
- Tracking validation loss helps monitor generalization during training.
- Loss curves help you diagnose training behavior visually.


# Task 4: Evaluating the model

### What you are learning
You are learning how to evaluate the neural network against a simpler baseline (Linear Regression), compare MSE values, and use scatter plots to visually inspect prediction quality.

### Steps (from the lab)
1. Import `LinearRegression` and `mean_squared_error`.
2. Train a baseline Linear Regression model.
3. Predict with both models.
4. Compute test MSE for both models.
5. Print the comparison.
6. Visualize predictions vs. true values.
   - 6.1 Create the figure
   - 6.2 Linear Regression scatter plot
   - 6.3 Neural Network scatter plot
   - 6.4 Interpret the plots


In [ ]:
# 1. Import baseline regression model and evaluation metric
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 2. Train baseline Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# 3. Predict with both models
y_pred_lr = lr_model.predict(X_test_scaled)

model.eval()
with torch.no_grad():
    y_pred_nn = model(X_test_tensor).numpy()


In [ ]:
# 4. Compute test MSE for both models
baseline_mse = mean_squared_error(y_test, y_pred_lr)
nn_mse = mean_squared_error(y_test, y_pred_nn)

# 5. Print comparison
print(f"Linear Regression Test MSE: {baseline_mse:.4f}")
print(f"Neural Network Test MSE:   {nn_mse:.4f}")


In [ ]:
# 6. Visualize predictions vs true values
# 6.1 Create figure
plt.figure(figsize=(12, 5))

# 6.2 Linear Regression plot
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_lr, alpha=0.6, color='blue')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Linear Regression Predictions')

# 6.3 Neural Network plot
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_nn, alpha=0.6, color='green')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('Neural Network Predictions')

plt.tight_layout()
plt.show()

# 6.4 Interpretation guide:
# Tighter alignment to a diagonal trend suggests better predictions.
# Spread/outliers indicate prediction error and where tuning may help.


### Task 4 explanation (learning takeaways)
- Baselines are essential for judging whether a neural network is truly improving performance.
- MSE gives a clear numeric error comparison.
- Scatter plots reveal structure that metrics alone may hide (bias, outliers, spread, nonlinear behavior).


# Task 5: Saving the trained model

### What you are learning
You are learning how to save a trained PyTorch model using `torch.save()` and `state_dict()`. This is the standard approach for reusing, sharing, or deploying trained models.

### Steps (from the lab)
1. Save model weights to file.
2. Confirm saving.
3. Review the confirmation output.


In [ ]:
# 1. Save model weights to file
torch.save(model.state_dict(), 'housing_mlp_model.pth')

# 2. Confirm saving
print("Model saved as 'housing_mlp_model.pth'")

# 3. Review the output above for confirmation


# Optional Extension: Loading the saved model (supports the overview topic)

### What you are learning
The project overview mentions saving and loading trained models. The original lab task explicitly demonstrates saving; this optional extension shows the matching load workflow using the same model architecture.


In [ ]:
# Optional: Load the saved model weights into a fresh model instance
loaded_model = HousingMLP()
loaded_model.load_state_dict(torch.load('housing_mlp_model.pth', map_location='cpu'))
loaded_model.eval()

print("Loaded model successfully and set to eval mode.")
print(loaded_model)


# Lab review

### What you are learning
This review checks your understanding of feature normalization and the evaluation approach used in the mini project.

## 1) Which of the following are valid reasons for normalizing the features before training a neural network model?
A. To reduce the dimensionality of the dataset  
B. To ensure all features contribute equally to the learning process  
C. To help the model converge faster during training  
D. To avoid converting the data into PyTorch tensors  
E. To remove irrelevant features from the dataset  

## 2) Which of the following statements correctly describe the model evaluation approach used in this lab?
A. Validation loss was used during training to monitor model generalization.  
B. The final model was evaluated using accuracy as the metric.  
C. A baseline Linear Regression model was used for performance comparison.  
D. Scatter plots were used to visually compare predicted and true values.  
E. Training loss was used to directly compare models.  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answers (click to expand)</strong></summary>

### 1) Normalizing features
Correct answers: **B, C**

### 2) Evaluation approach
Correct answers: **A, C, D**

</details>
